# Notebook 02 — Causal vs Hawkes disagreement

Plots R9 statistical α/μ against R10 IV-adjusted ATE per (leader,
pool_class). The (leader, pool) pairs in the upper-right quadrant
(high Hawkes, low causal ATE) are the news-confounding cases the
R10 causal gate would have suppressed.

_Spec_: `docs/ROUND_13_CALIBRATION_AND_RESEARCH.md` § 3.5


## Setup (re-uses 00_data_loader)


In [ ]:
%run 00_data_loader.ipynb


In [ ]:
import matplotlib.pyplot as plt, pandas as pd

df = con.execute('''
    SELECT leader_wallet,
           pool_class,
           hawkes_alpha_mu_ratio AS hawkes,
           causal_ate AS causal,
           causal_ate_ci_low, causal_ate_ci_high,
           wu_hausman_p, first_stage_f
    FROM causal_estimates
    WHERE hawkes_alpha_mu_ratio IS NOT NULL
      AND causal_ate IS NOT NULL
''').fetchdf()

if df.empty:
    print('NO DATA — populate via the R10 nightly causal daemon.')
else:
    print(f'Pairs: {len(df)}')
    plt.figure(figsize=(8, 8))
    plt.scatter(df.hawkes, df.causal, alpha=0.4)
    plt.plot([0, df.hawkes.max()], [0, df.hawkes.max()], 'r--', label='hawkes = causal')
    plt.xlabel('Hawkes α/μ (R5/R9)')
    plt.ylabel('Causal ATE (R10 IV-adjusted)')
    plt.title('Causal vs Hawkes — disagreement surfaces confounding')
    plt.legend(); plt.grid(alpha=0.3); plt.show()
